# Código para generar ontología de películas para probar métricas de similitud

## Cuidado!!! La ejecución tarda casi hora y media porque se hacen más de 700 consultas a wikidata. 

## No hace falta ejecutar. El resultado es la ontología: "ontologia_generos_cinematograficos_completa.ttl"

In [3]:
import sbc_tools as sbc

In [49]:
import time
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

class FilmGenreOntology:
    """
    Clase para construir una ontología RDF de géneros cinematográficos desde Wikidata.
    """
    
    def __init__(self):
        self.sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
        self.sparql.setReturnFormat(JSON)
        self.graph = Graph()
        
        # Definir namespaces
        self.WD = Namespace("http://www.wikidata.org/entity/")
        self.WDT = Namespace("http://www.wikidata.org/prop/direct/")
        self.FILM = Namespace("http://example.org/film-genre/")
        
        # Bind namespaces al grafo
        self.graph.bind("wd", self.WD)
        self.graph.bind("wdt", self.WDT)
        self.graph.bind("film", self.FILM)
        self.graph.bind("owl", OWL)
        self.graph.bind("rdfs", RDFS)
    
    def execute_query(self, query):
        """Ejecuta una consulta SPARQL en Wikidata"""
        time.sleep(7)  # Respetar límites de consulta
        try:
            self.sparql.setQuery(query)
            results = self.sparql.query().convert()
            return results["results"]["bindings"]
        except Exception as e:
            print(f"Error ejecutando consulta: {e}")
            return []
    
    def build_genre_taxonomy_from_wikidata(self, max_depth: int = None) -> Graph:
        """
        Consulta Wikidata y construye una ontología RDF con la jerarquía de géneros.
        
        Args:
            max_depth: Profundidad máxima del árbol de géneros. 
                      Si es None, obtiene toda la jerarquía sin límite.
                      Si es un entero, limita la profundidad a ese número de niveles.
                      Ejemplo: max_depth=3 genera wdt:P279/wdt:P279?/wdt:P279?
        
        Returns:
            Graph: Grafo RDF con la ontología de géneros cinematográficos
        """
        
        # Construir el property path según el parámetro de profundidad
        if max_depth is None:
            depth_path = "wdt:P279*"
            depth_info = "sin límite"
        else:
            # Primer nivel obligatorio, resto opcionales
            # max_depth=1: wdt:P279
            # max_depth=2: wdt:P279/wdt:P279?
            # max_depth=3: wdt:P279/wdt:P279?/wdt:P279?
            depth_path = "wdt:P279"
            for i in range(1, max_depth):
                depth_path += "/wdt:P279?"
            depth_info = f"hasta profundidad {max_depth}"
        
        # Consulta SPARQL para obtener la jerarquía de géneros con profundidad limitada
        query = f"""
        SELECT DISTINCT ?genre ?genreLabel ?parent ?parentLabel WHERE {{
          # Obtener todas las subclases de "film" (Q11424) con profundidad limitada
          ?genre {depth_path} wd:Q11424.
          
          # Obtener la relación directa de subclase
          ?genre wdt:P279 ?parent.
          ?parent wdt:P279* wd:Q11424.
          
          ?genre rdfs:label ?genreLabel
          FILTER (LANG(?genreLabel) = 'en')

          ?parent rdfs:label ?parentLabel
          FILTER (LANG(?parentLabel) = 'en')   
        }}
        ORDER BY ?genreLabel
        """
        
        print(f"Consultando Wikidata para obtener jerarquía de géneros ({depth_info})...")
        print("=" * 60)
        
        # Ejecutar la consulta
        bindings = self.execute_query(query)
        
        if bindings:
            # Construir el grafo RDF a partir de los resultados
            self._build_rdf_graph(bindings)
            print(f"\n✓ Ontología construida con {len(self.graph)} triples")
        else:
            print("No se obtuvieron resultados de Wikidata")
        
        return self.graph


    
    def _build_rdf_graph(self, bindings: list) -> None:
        """
        Construye el grafo RDF a partir de los resultados de la consulta SPARQL.
        
        Args:
            bindings: Lista de resultados de la consulta SPARQL
        """
        print(f"\nProcesando {len(bindings)} resultados...")
        
        # Añadir definición de la clase raíz
        film_class = self.WD.Q11424
        self.graph.add((film_class, RDF.type, OWL.Class))
        self.graph.add((film_class, RDFS.label, Literal("Film", lang="en")))
        self.graph.add((film_class, RDFS.comment, Literal("Root class for film genres", lang="en")))
        
        # Conjuntos para tracking
        self.genres_added = set()
        
        # Procesar cada resultado
        for i, binding in enumerate(bindings):
            # Extraer información del género
            genre_uri = URIRef(binding['genre']['value'])
            genre_label = binding['genreLabel']['value']
            # Extraer información del padre
            parent_uri = URIRef(binding['parent']['value'])
            parent_label = binding.get('parentLabel', {}).get('value', '')
            
            # Añadir el género como una clase OWL
            if genre_uri not in self.genres_added:
                self.graph.add((genre_uri, RDF.type, OWL.Class))
                self.graph.add((genre_uri, RDFS.label, Literal(genre_label, lang="en")))
                self.genres_added.add(genre_uri)
            
            # Añadir el padre como una clase OWL
            if parent_uri not in self.genres_added:
                self.graph.add((parent_uri, RDF.type, OWL.Class))
                if parent_label:
                    self.graph.add((parent_uri, RDFS.label, Literal(parent_label, lang="en")))
                self.genres_added.add(parent_uri)
            
            # Añadir la relación de subclase (rdfs:subClassOf)
            self.graph.add((genre_uri, RDFS.subClassOf, parent_uri))
            
        print(f"Procesamiento completado")

    def get_film_instances(self, genre_uri: str, n: int = 5) -> list:
        """
        Obtiene N instancias (películas) de un género específico con sus propiedades.
        
        Args:
            genre_uri: URI del género en Wikidata (ej: http://www.wikidata.org/entity/Q20443008)
            n: Número de instancias a obtener
            
        Returns:
            Lista de diccionarios con la información de cada película
        """
        # Extraer el ID del género de la URI
        genre_id = genre_uri.split('/')[-1]
        
        # Consulta SPARQL para obtener películas del género con propiedades obligatorias
        query = f"""
        SELECT DISTINCT ?film ?filmLabel ?imdbId 
               (SAMPLE(?countryLabel) as ?countryLabel)
               (GROUP_CONCAT(DISTINCT ?directorLabel; separator=";") AS ?directorLabels)
               (GROUP_CONCAT(DISTINCT ?castMemberLabel; separator=";") AS ?castMemberLabels)
        WHERE {{
          # Obtener películas que son instancias del género
          ?film wdt:P136 wd:{genre_id}.
          
          # Propiedades obligatorias
          ?film wdt:P57 ?director.          # director
          ?film wdt:P161 ?castMember.       # cast member
          ?film wdt:P495 ?country.          # country of origin
          ?film wdt:P345 ?imdbId.           # IMDb ID
          
          ?director rdfs:label ?directorLabel.
          FILTER(LANG(?directorLabel) = 'en')
          
          ?castMember rdfs:label ?castMemberLabel.
          FILTER(LANG(?castMemberLabel) = 'en')
          
          ?country rdfs:label ?countryLabel.
          FILTER(LANG(?countryLabel) = 'en')
          
          # Etiquetas en inglés
          SERVICE wikibase:label {{ 
            bd:serviceParam wikibase:language "en". 
          }}
        }}
        GROUP BY ?film ?filmLabel ?imdbId
        LIMIT {n}
        """
        
        return self.execute_query(query)
    
    def add_instances_to_ontology(self, n_per_genre: int = 5) -> None:
        """
        Añade instancias (películas) de cada género a la ontología.
        
        Args:
            n_per_genre: Número de películas a obtener por cada género
        """
        print("\n" + "=" * 60)
        print(f"OBTENIENDO INSTANCIAS DE PELÍCULAS ({n_per_genre} por género)")
        print("=" * 60)
        
        total_films = 0
        genres_processed = 0
        
        # Iterar sobre todos los géneros añadidos
        for genre_uri in self.genres_added:
            genres_processed += 1
            genre_label = self.graph.value(genre_uri, RDFS.label)
            
            print(f"\n[{genres_processed}/{len(self.genres_added)}] Consultando películas de: {genre_label}")
            
            # Obtener instancias del género
            film_instances = self.get_film_instances(str(genre_uri), n_per_genre)
            
            if film_instances:
                # Añadir las instancias al grafo
                films_added = self._add_films_to_graph(film_instances, genre_uri)
                total_films += films_added
                print(f"  ✓ Añadidas {films_added} películas")
            else:
                print(f"  ⚠ No se encontraron películas para este género")
        
        print("\n" + "=" * 60)
        print(f"✓ Total de películas añadidas: {total_films}")
        print(f"✓ Géneros procesados: {genres_processed}")
        print("=" * 60)
    
    def _add_films_to_graph(self, film_instances: list, genre_uri: URIRef) -> int:
        """
        Añade las instancias de películas al grafo RDF.
        
        Args:
            film_instances: Lista de resultados de películas de Wikidata
            genre_uri: URI del género al que pertenecen las películas
            
        Returns:
            Número de películas añadidas
        """
        films_added = 0
        
        for film_data in film_instances:
            try:
                # Extraer información de la película
                film_uri = URIRef(film_data['film']['value'])
                film_label = film_data['filmLabel']['value']
                
                # Añadir la película como instancia del género
                self.graph.add((film_uri, RDF.type, genre_uri))
                self.graph.add((film_uri, RDFS.label, Literal(film_label, lang="en")))
                
                # Añadir IMDb ID
                if 'imdbId' in film_data:
                    imdb_id = film_data['imdbId']['value']
                    self.graph.add((film_uri, self.WDT.P345, Literal(imdb_id)))
                
                # Añadir country of origin (un solo valor)
                if 'countryLabel' in film_data:
                    country_label = film_data['countryLabel']['value']
                    self.graph.add((film_uri, self.FILM.country, Literal(country_label, lang="en")))
                
                # Añadir directores (múltiples valores separados por ;)
                if 'directorLabels' in film_data:
                    directors = film_data['directorLabels']['value']
                    # Separar los directores y añadirlos individualmente
                    for director in directors.split(';'):
                        director = director.strip()
                        if director:
                            self.graph.add((film_uri, self.FILM.director, Literal(director, lang="en")))
                
                # Añadir cast members (múltiples valores separados por ;)
                if 'castMemberLabels' in film_data:
                    cast_members = film_data['castMemberLabels']['value']
                    # Separar los cast members y añadirlos individualmente
                    for cast_member in cast_members.split(';'):
                        cast_member = cast_member.strip()
                        if cast_member:
                            self.graph.add((film_uri, self.FILM.castMember, Literal(cast_member, lang="en")))
                
                films_added += 1
                
            except Exception as e:
                print(f"  ⚠ Error procesando película: {e}")
                continue
        
        return films_added


In [50]:
film_onto = FilmGenreOntology()
rdf_graph = film_onto.build_genre_taxonomy_from_wikidata(max_depth=3)

Consultando Wikidata para obtener jerarquía de géneros (hasta profundidad 3)...

Procesando 869 resultados...
Procesamiento completado

✓ Ontología construida con 2307 triples


In [34]:
sbc.show_graph(rdf_graph, "taxonomia_generos_cinematograficos.html")

In [51]:
film_onto.add_instances_to_ontology()


OBTENIENDO INSTANCIAS DE PELÍCULAS (5 por género)

[1/718] Consultando películas de: vigilante film
  ✓ Añadidas 5 películas

[2/718] Consultando películas de: Lek Chalali Sasarla
  ⚠ No se encontraron películas para este género

[3/718] Consultando películas de: anthology film
Error ejecutando consulta: HTTP Error 504: Gateway Timeout
  ⚠ No se encontraron películas para este género

[4/718] Consultando películas de: news cinema
  ⚠ No se encontraron películas para este género

[5/718] Consultando películas de: medium-length film
  ⚠ No se encontraron películas para este género

[6/718] Consultando películas de: New Nigerian Cinema
  ⚠ No se encontraron películas para este género

[7/718] Consultando películas de: crime film
Error ejecutando consulta: HTTP Error 429: Too Many Requests
  ⚠ No se encontraron películas para este género

[8/718] Consultando películas de: Chronophone
  ⚠ No se encontraron películas para este género

[9/718] Consultando películas de: film based on a audiov

In [ ]:
sbc.save(film_onto.graph, "ontologia_generos_cinematograficos_completa.ttl", format="turtle")

<Graph identifier=Ned1ad10952054ff3af67d4ffeff534fa (<class 'rdflib.graph.Graph'>)>

In [53]:
sbc.show_graph(rdf_graph, "ontologia_generos_cinematograficos_completa.html")

---
# Ejercicio

## Extiende tu grafo de conocimiento para que tenga una jerarquía con distintos niveles